In [1]:
import pandas as pd
import requests
import requests as req
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)
import matplotlib.pyplot as plt
from io import StringIO
import lxml as lxml
import yfinance as yf
from pandas import read_html
import numpy as np




def make_graph(stock_data, revenue_data, stock):
    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30']

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Stock price
    axes[0].plot(pd.to_datetime(stock_data_specific.Date), stock_data_specific.Close.astype("float"), label="Share Price", color="blue")
    axes[0].set_ylabel("Price ($US)")
    axes[0].set_title(f"{stock} - Historical Share Price")

    # Revenue
    axes[1].plot(pd.to_datetime(revenue_data_specific.Date), revenue_data_specific.Revenue.astype("float"), label="Revenue", color="green")
    axes[1].set_ylabel("Revenue ($US Millions)")
    axes[1].set_xlabel("Date")
    axes[1].set_title(f"{stock} - Historical Revenue")

    plt.tight_layout()
    plt.show()



url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"

TeslaData= yf.Ticker("TSLA")
periodData = TeslaData.history(period="max")
periodData.reset_index(inplace=True)
print(periodData.head())
periodData.plot(x="Date", y="Open")

GameStopData = yf.Ticker("GME")
GameStopPeriodData = GameStopData.history(period="max")

GameStopPeriodData.reset_index(inplace=True)
print(GameStopPeriodData.head())

try:
    data = req.get(url).text
    soup = BeautifulSoup(data, 'html.parser')
    tesla_data = pd.DataFrame(columns=["Date", "Revenue"])

    for row in soup.find("tbody").find_all('tr'):
        col = row.find_all("td")
        Date = col[0].text
        Revenue = col[1].text

        # Finally we append the data of each row to the table
        tesla_data = pd.concat([tesla_data , pd.DataFrame(
            {"Date": [Date], "Revenue": [Revenue]})], ignore_index=True)
    read_html_pandas_data = pd.read_html(StringIO(str(soup)))
    tesla_dataframe = read_html_pandas_data[0]

    print(tesla_data.tail())

    gmeurl = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
    gmedata = requests.get(gmeurl).text
    gmesoup = BeautifulSoup(gmedata, 'html.parser')
    gme_data = pd.DataFrame(columns=["Date", "Revenue"])
    for gmerow in gmesoup.find("tbody").find_all('tr'):
        column = gmerow.find_all("td")
        Date = column[0].text
        Revenue = column[1].text
        gme_data = pd.concat([gme_data, pd.DataFrame(
            {"Date": [Date], "Revenue": [Revenue]}
        )], ignore_index=True)
        read_html_gme_data = pd.read_html(StringIO(str(gmesoup)))
        gme_dataframe = read_html_gme_data[0]
    print(gme_data.tail())

except Exception as e:
      print(e)

tesla_data["Revenue"] = tesla_data["Revenue"].str.replace(",", "").str.replace("$", "")
make_graph(periodData, tesla_data, "Tesla")

gme_data["Revenue"] = gme_data["Revenue"].str.replace(",", "").str.replace("$", "")
make_graph(GameStopPeriodData,gme_data, "GameStop" )

ModuleNotFoundError: No module named 'yfinance'